# Caso 1 · El dashboard vende más que finanzas
**Guiado · 20 minutos**

> **De:** Dirección General
> **Para:** BI, Finanzas
> **Asunto:** ¿Cuánto vendimos en agosto?
>
> Comercial me presenta **$2,097,636.02** y Finanzas **$2,024,187.62**.
> Mañana es el comité. ¿Qué número llevo y por qué son distintos?

**Telar & Barro** (empresa ficticia) vende artesanías yucatecas en línea y en tres sucursales.

| Tabla | Un renglón es… | Notas del equipo de datos |
|---|---|---|
| `pedidos` | un pedido | `fecha_pedido`: fecha y hora del pedido. `estatus`: pagado / pendiente |
| `lineas_pedido` | un producto dentro de un pedido | `precio_unitario` sin IVA |
| `pagos` | un pago | `monto` sin IVA. `pagado_en` con zona horaria |
| `productos`, `clientes` | catálogo | |

**El método** (lo usaremos en los tres casos):
1. Confirmar y acotar · 2. Comparar definiciones · 3. Comparar en el mismo grano y contar filas · 4. Partir en dos · 5. Aislar registros · 6. Explicar y demostrar · 7. Volverlo prueba

In [ ]:
# ⚙️ Preparación: ejecuta esta celda primero (funciona en Colab y en local)
import os, sys, subprocess, pathlib
REPO_URL = "https://github.com/jona866/taller-conciliacion-de-datos-datostada2026.git"  # ← el facilitador lo actualiza
if "google.colab" in sys.modules:
    destino = pathlib.Path("/content/taller-conciliacion-de-datos-datostada2026")
    if not destino.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(destino)], check=True)
    os.chdir(destino)
else:
    raiz = pathlib.Path.cwd()
    while not (raiz / "nnc.py").exists() and raiz != raiz.parent:
        raiz = raiz.parent
    os.chdir(raiz)
sys.path.insert(0, os.getcwd())
import nnc
con = nnc.conectar("caso1")
print("Listo ✅  Tablas:", ", ".join(t for (t,) in con.execute("SHOW TABLES").fetchall()))

## Paso 1 · Confirmar y acotar
Antes de buscar errores, **reproduce las dos cifras con el SQL real** de cada reporte. Si no puedes reproducirlas, no puedes explicarlas.

In [ ]:
comercial = nnc.total(con, "caso1_dashboard_ventas")
finanzas = nnc.total(con, "caso1_reporte_finanzas")
print(f"Comercial : {comercial:>14,.2f}")
print(f"Finanzas  : {finanzas:>14,.2f}")
print(f"Diferencia: {comercial - finanzas:>14,.2f}")

¿La diferencia está en todo el mes o en ciertos días? Compara **por día**.

In [ ]:
dash = nnc.reporte(con, "caso1_dashboard_ventas")
fin = nnc.reporte(con, "caso1_reporte_finanzas")
por_dia = dash.merge(fin, on="dia", how="outer").fillna(0)
por_dia["diferencia"] = (por_dia["ventas"] - por_dia["ingresos"]).round(2)
print("Días con diferencia:", (por_dia["diferencia"] != 0).sum(), "de", len(por_dia))
print("Suma de días con comercial > finanzas:", round(por_dia.loc[por_dia.diferencia > 0, "diferencia"].sum(), 2))
print("Suma de días con finanzas > comercial:", round(por_dia.loc[por_dia.diferencia < 0, "diferencia"].sum(), 2))
por_dia[por_dia["diferencia"] != 0]

**Pregunta:** hay días en que finanzas es mayor. Esa parte se compensa dentro del mes. ¿Qué tipo de diferencia es?

<details><summary>💡 Pista</summary>

Si una diferencia aparece un día y desaparece otro, casi siempre es de **momento** (qué fecha usa cada reporte), no de **monto**. Guárdala para el paso 2. Lo que no se compensa es lo que hay que explicar.

</details>

## Paso 2 · Comparar definiciones
Lee el SQL de cada reporte como si fuera un contrato.

In [ ]:
nnc.ver_sql("caso1_dashboard_ventas")
print("=" * 70)
nnc.ver_sql("caso1_reporte_finanzas")

Completa (puede ser mentalmente):

| | Comercial | Finanzas |
|---|---|---|
| ¿Qué suma? | | |
| ¿Qué fecha usa? | | |
| ¿Qué filtra? | | |
| ¿Con qué tablas une? | | |

## Paso 3 · Mismo grano: cuenta filas antes y después de cada JOIN
El **grano** es lo que representa un renglón. `lineas_pedido`: un producto en un pedido. `pagos`: un pago.
Si unes dos tablas con granos distintos, cada línea se repite tantas veces como pagos tenga su pedido.

In [ ]:
antes = con.execute("""
    SELECT COUNT(*)
    FROM pedidos p
    JOIN lineas_pedido l ON l.pedido_id = p.pedido_id
    WHERE p.estatus = 'pagado'
      AND CAST(p.fecha_pedido AS DATE) BETWEEN DATE '2026-08-01' AND DATE '2026-08-31'
""").fetchone()[0]
print("Líneas de pedidos pagados en agosto:", antes)

In [ ]:
# ✍️ Tu turno: escribe tu consulta aquí
# Agrega el JOIN con pagos (igual que el dashboard) y vuelve a contar.
despues = None

nnc.revisar("caso1.filas_despues", despues)

In [ ]:
# ✍️ Tu turno: escribe tu consulta aquí
# ¿Cuántos pedidos tienen más de un pago? (GROUP BY ... HAVING)
multipago = nnc.q(con, """
    -- tu consulta
    SELECT 1
""")
multipago

## Pasos 4 y 5 · Partir en dos y aislar
Separa los pedidos con **un** pago de los que tienen **varios**. ¿Dónde vive la diferencia?

In [ ]:
nnc.q(con, """
WITH n_pagos AS (SELECT pedido_id, COUNT(*) AS n FROM pagos GROUP BY 1),
lineas AS (
    SELECT p.pedido_id, SUM(l.cantidad * l.precio_unitario) AS valor
    FROM pedidos p JOIN lineas_pedido l USING (pedido_id)
    GROUP BY 1
)
SELECT CASE WHEN n.n = 1 THEN 'un pago' ELSE 'varios pagos' END AS grupo,
       COUNT(*)                         AS pedidos,
       SUM(l.valor * n.n)               AS como_lo_suma_el_dashboard,
       SUM(l.valor)                     AS valor_real,
       SUM(l.valor * n.n) - SUM(l.valor) AS diferencia
FROM lineas l JOIN n_pagos n USING (pedido_id)
GROUP BY 1
""")

## Paso 6 · Explicar y demostrar
Corrige la consulta del dashboard **sin cambiar su intención** (solo pedidos pagados).

In [ ]:
# ✍️ Tu turno: escribe tu consulta aquí
# Pista: EXISTS (SELECT 1 FROM pagos g WHERE g.pedido_id = p.pedido_id)
corregido = con.execute("""
    SELECT 0
""").fetchone()[0]

print(f"Comercial corregido: {corregido:,.2f}  ·  Finanzas: {finanzas:,.2f}")
nnc.revisar("caso1.diferencia", float(comercial) - float(corregido))

**Trampa común:** “si el JOIN duplica, lo quito y ya”. Mira qué pasa:

In [ ]:
sin_join = con.execute("""
    SELECT SUM(l.cantidad * l.precio_unitario)
    FROM pedidos p JOIN lineas_pedido l ON l.pedido_id = p.pedido_id
    WHERE CAST(p.fecha_pedido AS DATE) BETWEEN DATE '2026-08-01' AND DATE '2026-08-31'
""").fetchone()[0]
pendientes = con.execute("SELECT COUNT(*) FROM pedidos WHERE estatus <> 'pagado'").fetchone()[0]
print(f"Sin el JOIN: {sin_join:,.2f} → ahora sobran {sin_join - finanzas:,.2f}")
print(f"Se colaron {pendientes} pedidos en línea abandonados (sin pago). Arreglaste un bug y metiste otro.")

**Explica la causa en dos frases**, como se la dirías a Dirección:

> ...

## Paso 7 · Volverlo prueba
Una prueba es una afirmación que debe ser cierta **siempre**. Esta habría avisado antes del comité.

In [ ]:
def filas(sql):
    return con.execute(sql).fetchone()[0]

base = "FROM pedidos p JOIN lineas_pedido l ON l.pedido_id = p.pedido_id"
lineas_pagadas = filas(f"SELECT COUNT(*) {base} WHERE EXISTS (SELECT 1 FROM pagos g WHERE g.pedido_id = p.pedido_id)")
filas_dashboard_original = filas(f"SELECT COUNT(*) {base} JOIN pagos g ON g.pedido_id = p.pedido_id")

nnc.probar("Dashboard original: el JOIN no multiplica filas", filas_dashboard_original == lineas_pagadas,
           f"{filas_dashboard_original} filas vs {lineas_pagadas} líneas")
nnc.probar("Dashboard corregido = finanzas", abs(float(corregido) - float(finanzas)) < 0.01)

### Lo que te llevas del caso 1
- Cuenta filas **antes y después** de cada JOIN. Si cambia el conteo, cambió el grano.
- Una diferencia que se compensa entre días es de **fecha**; la que no se compensa es de **monto**.
- El arreglo debe conservar la intención del reporte (por eso `EXISTS` y no “quitar el JOIN”).